In [14]:
import os 
from dotenv import load_dotenv
import glob
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [2]:
MODEL = "llama-3.1-8b-instant"
db_name = "vector_db"
load_dotenv(override=True)

True

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2192.60it/s]


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatGroq(temperature=0, model_name=MODEL)

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='f03b44c6-3152-4179-ae1d-db138bab0c57', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='There are several notable individuals named Avery. Here are a few:\n\n1. **Avery Bradley**: An American professional basketball player who plays in the NBA. He has played for several teams, including the Boston Celtics and Los Angeles Clippers.\n2. **Avery Bradley (musician)**: An American musician and producer who has worked with various artists, including Kendrick Lamar and J. Cole.\n3. **Avery Bradley (footballer)**: An American soccer player who has played for several teams, including the Seattle Sounders FC.\n4. **Avery Bradley (author)**: An American author who has written several books, including "The Girl Who Drank the Moon" and "The Vanderbeekers of 141st Street".\n5. **Avery Bradley (TV personality)**: An American TV personality who has appeared on several reality TV shows, including "The Bachelor" and "The Bachelorette".\n6. **Avery Bradley (music)**: Avery Bradley is also a music artist, who has released several albums and singles.\n\nHowever, without mor

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def get_answers(question: str,history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
    return response.content

In [13]:
get_answers("Who is Avery?",[])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been instrumental in guiding the company to its current position as a leading Insurance Tech provider.'

In [16]:
gr.ChatInterface(get_answers).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [18]:
gr.close_all()

Closing server running on port: 7860


TWO PROBS: 
- history not passed to llm ,not maintaining conversation history
- looking upto the context of recent messages only